**Reformat Base**

In [6]:
def extraer_formatos(path_sas):
    formatos = {}
    formato_actual = None
    leyendo_formato = False

    with open(path_sas, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()

            # Inicia nuevo bloque de formato
            if linea.lower().startswith("value"):
                partes = linea.split()
                if len(partes) >= 2:
                    formato_actual = partes[1].replace(";", "").replace("(", "").strip().upper()
                    formatos[formato_actual] = {}
                    leyendo_formato = True
                continue

            # Si estamos leyendo un formato y encontramos un "fin de bloque"
            if leyendo_formato and linea == ";":
                leyendo_formato = False
                formato_actual = None
                continue

            # Si estamos leyendo un bloque de formato, procesa las líneas tipo: 1 = "Texto"
            if leyendo_formato and "=" in linea:
                try:
                    clave, valor = linea.split("=", 1)
                    clave = clave.strip()
                    # Maneja rangos simples como 1 - 3 = "Bajo" (si fuera necesario puedes ampliar esto)
                    if "–" in clave or "-" in clave:
                        # Rango no soportado aún (puede añadirse después si lo necesitas)
                        continue
                    clave = int(clave)
                    valor = valor.strip().strip('"').strip("'")
                    formatos[formato_actual][clave] = valor
                except Exception:
                    continue
    return formatos


In [7]:
# formatos = extraer_formatos("../STU_COG_SAS/CY08MSP_STU_COG.FORMAT.SAS")
formatos = extraer_formatos("../data/2018/collection_sas/SAS_STU_COG/COG/cy07_msu_stu_cog.sas7bdat")
formatos

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 12: invalid continuation byte

In [8]:
print("Claves disponibles en formatos:", formatos.keys())

Claves disponibles en formatos: dict_keys(['ADMINMODEF', 'BOOKIDF', 'CMA112Q02WDF', 'CMA116Q02SF', 'CMA125Q01SF', 'CMA125Q02SF', 'CMA129Q02SF', 'CMA152Q02SF', 'CMA157Q01SF', '$', 'CNTRYIDF', 'CR561Q06SF', 'CR570Q04SDF', 'CS646Q04WF', 'CS657Q04TTF', 'DM155Q02CF', 'DM462Q01CF', 'DM474Q01RF', 'DM936Q02CF', 'DM953Q04CF', 'DMA103Q05RAF', 'DMA119Q03CF', 'DMA128Q03RCF', 'DMA136Q02CF', 'DMA152Q02RAF', 'DMA161Q04CF', 'DMA162Q02RCF', 'DR055Q03CF', 'DR590Q66RF', 'DS438Q03CF', 'DS635Q04RBF', 'DS635Q05CF', 'DS643Q01RF', 'DS645Q01REF', 'DS645Q03RCF', 'DS648Q01CF', 'DS648Q02RAF', 'DS657Q03RF', 'DS657Q04CF', 'LANGTEST_COGF', 'MPATHF', 'MS1_LEVF', 'MS2_ITEMSETF', 'MS2_LEVF', 'MS2_TESTF', 'OPTION_UHF', 'PM155Q02F', 'PM155Q03F', 'PM192Q01CF', 'PM273Q01DF', 'PM462Q01F', 'PM803Q01F', 'PR219Q01SF', 'PR227Q02SF', 'PR412Q01F', 'PR432Q06IF', 'PR921P02AF', 'PR921P03AF', 'PR921P04AF', 'PR921P05AF', 'PR921P07AF', 'PR921P08AF', 'PR921P09AF', 'PR921P10AF', 'PR921P11AF', 'PR921P12AF', 'PR921P35AF', 'PR921P36AF', 'PR

In [4]:
# formatos = extraer_formatos("../STU_COG_SAS/CY08MSP_STU_COG.FORMAT.SAS")
formatos = extraer_formatos("BD_PISA_2022_Peru/BD_PISA_2022_Peru/CY08MSP_STU_COG.FORMAT.SAS")
adminmode_dict = formatos.get("ADMINMODEF", {})
bookid_dict = formatos.get("BOOKIDF", {})

**Transformar .SAS7BDAT a .CSV**

In [15]:
import pandas as pd
import pyreadstat
import re

def extraer_formatos_y_asignaciones(path_sas):
    formatos = {}
    asignaciones = {}
    formato_actual = None
    leyendo_formato = False
    leyendo_asignaciones = False

    with open(path_sas, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()

            # Detectar inicio de bloque de formato
            if linea.lower().startswith("value"):
                partes = linea.split()
                if len(partes) >= 2:
                    formato_actual = partes[1].replace(";", "").replace("(", "").strip().upper()
                    formatos[formato_actual] = {}
                    leyendo_formato = True
                continue

            # Fin del bloque de formato
            if leyendo_formato and linea == ";":
                leyendo_formato = False
                formato_actual = None
                continue

            # Dentro de bloque de formato
            if leyendo_formato and "=" in linea:
                try:
                    clave, valor = linea.split("=", 1)
                    clave = clave.strip()
                    if "-" in clave or "–" in clave:
                        continue  # ignorar rangos por ahora
                    clave = int(clave)
                    valor = valor.strip().strip('"').strip("'")
                    formatos[formato_actual][clave] = valor
                except Exception:
                    continue

            # Detectar bloque de asignaciones
            if linea.lower().startswith("format"):
                leyendo_asignaciones = True
                continue

            # Fin del bloque de asignaciones
            if leyendo_asignaciones and linea == ";":
                leyendo_asignaciones = False
                continue

            # Extraer variable → formato
            if leyendo_asignaciones and "." in linea:
                partes = re.findall(r"(\w+)\s+(\w+)\.", linea)
                for variable, formato in partes:
                    asignaciones[variable.upper()] = formato.upper()

    return formatos, asignaciones


In [ ]:
# Rutas
archivo_sas = "../STU_COG_SAS/CY08MSP_STU_COG.FORMAT.SAS"
archivo_datos = "../STU_COG_SAS/CY08MSP_STU_COG.SAS7BDAT"

# Leer formatos y asignaciones
formatos, asignaciones = extraer_formatos_y_asignaciones(archivo_sas)

# Leer datos
df, meta = pyreadstat.read_sas7bdat(archivo_datos)

# Aplicar formatos a las columnas que coincidan
for col in df.columns:
    col_upper = col.upper()
    if col_upper in asignaciones:
        nombre_formato = asignaciones[col_upper]
        diccionario = formatos.get(nombre_formato)
        if diccionario:
            df[col] = df[col].map(diccionario).fillna(df[col])

# Guardar resultado
df.to_csv("datos_formateados_CRT_COG_SAS.csv", index=False)

**Transformar .SAS7BDAT a .CSV por chunks**

* Nuevo Chunk - Usando de referencia antiguo transformador * 

In [4]:
import pandas as pd
import pyreadstat
import re
import os
import gc
import psutil
from typing import Dict, Tuple

In [5]:
def extraer_formatos_y_asignaciones(path_sas: str) -> Tuple[Dict, Dict]:
    """Extrae formatos y asignaciones del archivo SAS"""
    formatos = {}
    asignaciones = {}
    formato_actual = None
    leyendo_formato = False
    leyendo_asignaciones = False

    with open(path_sas, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()

            # Detectar inicio de bloque de formato
            if linea.lower().startswith("value"):
                partes = linea.split()
                if len(partes) >= 2:
                    formato_actual = partes[1].replace(";", "").replace("(", "").strip().upper()
                    formatos[formato_actual] = {}
                    leyendo_formato = True
                continue

            # Fin del bloque de formato
            if leyendo_formato and linea == ";":
                leyendo_formato = False
                formato_actual = None
                continue

            # Dentro de bloque de formato
            if leyendo_formato and "=" in linea:
                try:
                    clave, valor = linea.split("=", 1)
                    clave = clave.strip()
                    if "-" in clave or "–" in clave:
                        continue  # ignorar rangos por ahora
                    clave = int(clave)
                    valor = valor.strip().strip('"').strip("'")
                    formatos[formato_actual][clave] = valor
                except Exception:
                    continue

            # Detectar bloque de asignaciones
            if linea.lower().startswith("format"):
                leyendo_asignaciones = True
                continue

            # Fin del bloque de asignaciones
            if leyendo_asignaciones and linea == ";":
                leyendo_asignaciones = False
                continue

            # Extraer variable → formato
            if leyendo_asignaciones and "." in linea:
                partes = re.findall(r"(\w+)\s+(\w+)\.", linea)
                for variable, formato in partes:
                    asignaciones[variable.upper()] = formato.upper()

    return formatos, asignaciones

def obtener_memoria_disponible_gb():
    """Obtiene la memoria RAM disponible en GB"""
    memoria = psutil.virtual_memory()
    disponible_gb = memoria.available / (1024**3)
    return disponible_gb

def estimar_chunk_size_seguro(max_memoria_gb: float = 2.0) -> int:
    """Calcula un chunk size seguro basado en la memoria disponible"""
    memoria_disponible = obtener_memoria_disponible_gb()

    # Usar como máximo la mitad de la memoria disponible o el límite especificado
    memoria_a_usar = min(max_memoria_gb, memoria_disponible / 2)

    print(f"Memoria RAM disponible: {memoria_disponible:.1f} GB")
    print(f"Memoria que usaremos: {memoria_a_usar:.1f} GB")

    # Estimación conservadora: ~500 bytes por celda, ~5000 columnas
    # Chunk size = (memoria_en_bytes) / (columnas_estimadas * bytes_por_celda)
    bytes_por_celda = 500
    columnas_estimadas = 5000

    chunk_size = int((memoria_a_usar * 1024**3) / (columnas_estimadas * bytes_por_celda))

    # Limitar entre valores razonables
    chunk_size = max(1000, min(chunk_size, 100000))

    print(f"Chunk size calculado (conservador): {chunk_size:,} filas")
    return chunk_size

def aplicar_formatos_chunk(chunk_df: pd.DataFrame, formatos: Dict, asignaciones: Dict) -> pd.DataFrame:
    """Aplica formatos a un chunk específico"""
    print(f"  Aplicando formatos al chunk...")
    formatos_aplicados = 0

    for col in chunk_df.columns:
        col_upper = col.upper()
        if col_upper in asignaciones:
            nombre_formato = asignaciones[col_upper]
            diccionario = formatos.get(nombre_formato)
            if diccionario:
                try:
                    chunk_df[col] = chunk_df[col].map(diccionario).fillna(chunk_df[col])
                    formatos_aplicados += 1
                except Exception as e:
                    print(f"    Error aplicando formato a {col}: {e}")

    print(f"  Formatos aplicados: {formatos_aplicados}")
    return chunk_df

def procesar_archivo_por_chunks_seguro(archivo_sas: str, archivo_datos: str, archivo_salida: str, chunk_size: int = None):
    """Versión ultra-segura para procesar archivos grandes"""

    print("=== PROCESAMIENTO ULTRA-SEGURO POR CHUNKS ===")
    print(f"Archivo: {archivo_datos}")

    # 1. Extraer formatos y asignaciones
    print("\n1. Extrayendo formatos y asignaciones...")
    try:
        formatos, asignaciones = extraer_formatos_y_asignaciones(archivo_sas)
        print(f"   ✓ Formatos extraídos: {len(formatos)}")
        print(f"   ✓ Asignaciones extraídas: {len(asignaciones)}")
    except Exception as e:
        print(f"   ✗ Error extrayendo formatos: {e}")
        return False

    # 2. Determinar chunk size si no se especifica
    if chunk_size is None:
        chunk_size = estimar_chunk_size_seguro(max_memoria_gb=2.0)
    else:
        print(f"\nUsando chunk size especificado: {chunk_size:,} filas")

    # 3. Procesar archivo
    print(f"\n2. Iniciando procesamiento con chunks de {chunk_size:,} filas")

    archivo_temporal = archivo_salida.replace('.csv', '_temp.csv')
    primera_iteracion = True
    total_filas_procesadas = 0
    chunk_num = 0
    start_row = 0

    # Limpiar archivo temporal si existe
    if os.path.exists(archivo_temporal):
        os.remove(archivo_temporal)

    try:
        while True:
            chunk_num += 1
            print(f"\n--- Chunk {chunk_num} ---")
            print(f"Intentando leer desde fila {start_row:,}, máximo {chunk_size:,} filas")

            # Mostrar memoria antes de cargar
            memoria_antes = obtener_memoria_disponible_gb()
            print(f"Memoria disponible antes: {memoria_antes:.1f} GB")

            try:
                # Intentar leer el chunk
                if start_row == 0:
                    # Primera lectura, intentar con row_limit
                    chunk_df, meta = pyreadstat.read_sas7bdat(
                        archivo_datos,
                        row_limit=chunk_size
                    )
                else:
                    # Lecturas subsecuentes con offset
                    chunk_df, meta = pyreadstat.read_sas7bdat(
                        archivo_datos,
                        row_offset=start_row,
                        row_limit=chunk_size
                    )

                # Verificar si obtuvimos datos
                if len(chunk_df) == 0:
                    print("No hay más datos para procesar")
                    break

                filas_en_chunk = len(chunk_df)
                columnas_en_chunk = len(chunk_df.columns)
                memoria_chunk = chunk_df.memory_usage(deep=True).sum() / (1024**3)

                print(f"✓ Chunk cargado: {filas_en_chunk:,} filas × {columnas_en_chunk} columnas")
                print(f"✓ Memoria del chunk: {memoria_chunk:.2f} GB")

                # Mostrar memoria después de cargar
                memoria_despues = obtener_memoria_disponible_gb()
                print(f"Memoria disponible después: {memoria_despues:.1f} GB")

                # Aplicar formatos
                chunk_df = aplicar_formatos_chunk(chunk_df, formatos, asignaciones)

                # Guardar chunk
                print("  Guardando chunk...")
                if primera_iteracion:
                    chunk_df.to_csv(archivo_temporal, index=False, mode='w')
                    primera_iteracion = False
                    print("  ✓ Archivo CSV creado")
                else:
                    chunk_df.to_csv(archivo_temporal, index=False, mode='a', header=False)
                    print("  ✓ Chunk añadido al CSV")

                total_filas_procesadas += filas_en_chunk
                print(f"✓ Total procesado hasta ahora: {total_filas_procesadas:,} filas")

                # Limpiar memoria agresivamente
                del chunk_df
                if 'meta' in locals():
                    del meta
                gc.collect()

                memoria_final = obtener_memoria_disponible_gb()
                print(f"Memoria disponible después de limpiar: {memoria_final:.1f} GB")

                # Actualizar start_row para el siguiente chunk
                start_row += filas_en_chunk

                # Si obtuvimos menos filas de las solicitadas, hemos terminado
                if filas_en_chunk < chunk_size:
                    print("Último chunk procesado (menos filas de las solicitadas)")
                    break

            except MemoryError as e:
                print(f"✗ Error de memoria en chunk {chunk_num}: {e}")

                # Si el chunk size es muy pequeño y aún falla, es un problema serio
                if chunk_size <= 1000:
                    print("✗ CRÍTICO: Incluso con chunks de 1000 filas hay problemas de memoria")
                    return False

                # Reducir chunk size y reintentar desde la misma posición
                chunk_size = max(1000, chunk_size // 2)
                print(f"⚠ Reduciendo chunk size a {chunk_size:,} filas")
                gc.collect()
                continue

            except Exception as e:
                print(f"✗ Error inesperado en chunk {chunk_num}: {e}")

                # Si es el primer chunk, es un problema serio
                if chunk_num == 1:
                    print("✗ Error en el primer chunk, abortando")
                    return False

                # Para chunks posteriores, intentar continuar
                print("⚠ Intentando continuar con el siguiente chunk")
                start_row += chunk_size
                continue

        # Finalizar
        if os.path.exists(archivo_temporal) and total_filas_procesadas > 0:
            if os.path.exists(archivo_salida):
                os.remove(archivo_salida)
            os.rename(archivo_temporal, archivo_salida)

            print(f"\n=== ✅ PROCESAMIENTO COMPLETADO ===")
            print(f"📁 Archivo final: {archivo_salida}")
            print(f"📊 Total filas procesadas: {total_filas_procesadas:,}")
            print(f"📦 Total chunks procesados: {chunk_num}")

            # Información del archivo final
            if os.path.exists(archivo_salida):
                tamaño_mb = os.path.getsize(archivo_salida) / (1024**2)
                print(f"💾 Tamaño archivo final: {tamaño_mb:.1f} MB")

            return True
        else:
            print("✗ No se pudo crear el archivo de salida")
            return False

    except Exception as e:
        print(f"✗ Error crítico: {e}")
        return False

    finally:
        # Limpiar archivos temporales
        if os.path.exists(archivo_temporal):
            try:
                os.remove(archivo_temporal)
            except:
                pass

In [20]:
# ===== CONFIGURACIÓN Y EJECUCIÓN =====

if __name__ == "__main__":
    # Rutas de archivos
    # archivo_sas = "../STU_QQQ_SAS/CY08MSP_STU_QQQ.FORMAT.SAS"
    # archivo_datos = "../STU_QQQ_SAS/CY08MSP_STU_QQQ.SAS7BDAT"
    # archivo_salida = "../data_collection_csv/datos_formateados_CY08MSP_STU_QQQ.csv"

    # Rutas de archivos
    # archivo_sas = "../data/2018/collection_sas/SAS_STU_COG/COG/CY07MSU_STU_COG.FORMAT.SAS"
    # archivo_datos = "../data/2018/collection_sas/SAS_STU_COG/COG/cy07_msu_stu_cog.sas7bdat"
    # archivo_salida = "../data/2018/collection_csv/CY07MSU_STU_COG.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07_QMC_FLT_TTM.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07_QMC_FLT_TTM.SAS7BDAT"
    # archivo_salida  = "../data/2018/collection_csv/CY07_QMC_FLT_TTM.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07_QMC_STU_TTM.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07_QMC_STU_TTM.SAS7BDAT"
    # archivo_salida  = "../data/2018/collection_csv/CY07_QMC_STU_TTM.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_FLT_COG.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_FLT_COG.SAS7BDAT"
    # archivo_salida  = "../data/2018/collection_csv/CY07MSU_QMC_FLT_COG.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_FLT_QQQ.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_FLT_QQQ.SAS7BDAT"
    # archivo_salida  = "../data/2018/collection_csv/CY07MSU_QMC_FLT_QQQ.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_FLT_TIM.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_FLT_TIM.SAS7BDAT"
    # archivo_salida  = "../data/2018/collection_csv/CY07MSU_QMC_FLT_TIM.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_STU_COG.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/cy07msu_qmc_stu_cog.sas7bdat"
    # archivo_salida  = "../data/2018/collection_csv/CY07MSU_QMC_STU_COG.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_STU_QQQ.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/cy07msu_qmc_stu_qqq.sas7bdat"
    # archivo_salida  = "../data/2018/collection_csv/CY07MSU_QMC_STU_QQQ.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/CY07MSU_QMC_STU_TIM.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_QMC_ALLDB/QMC/cy07msu_qmc_stu_tim.sas7bdat"
    # archivo_salida  = "../data/2018/collection_csv/CY07MSU_QMC_STU_TIM.csv"

    # archivo_sas     = "../data/2018/collection_sas/SAS_SCH_QQQ/SCH/CY07MSU_SCH_QQQ.FORMAT.SAS"
    # archivo_datos   = "../data/2018/collection_sas/SAS_SCH_QQQ/SCH/cy07_msu_sch_qqq.sas7bdat"
    # archivo_salida  = "../data/2018/collection_csv/CY07MSU_SCH_QQQ.csv"

    archivo_sas     = "../data/2018/collection_sas/SAS_STU_FLT/FLT/CY07_MSU_FLT_TTM.FORMAT.SAS"
    archivo_datos   = "../data/2018/collection_sas/SAS_STU_FLT/FLT/CY07_MSU_FLT_TTM.SAS7BDAT"
    archivo_salida  = "../data/2018/collection_csv/CY07_MSU_FLT_TTM.csv"

    print("=== CONVERSOR SAS ULTRA-SEGURO ===")
    print("Diseñado específicamente para evitar errores de memoria")

    # Verificar archivos
    if not os.path.exists(archivo_sas):
        print(f"✗ No se encuentra: {archivo_sas}")
        exit(1)

    if not os.path.exists(archivo_datos):
        print(f"✗ No se encuentra: {archivo_datos}")
        exit(1)

    print("✓ Archivos encontrados")

    # Mostrar información del sistema
    memoria_total = psutil.virtual_memory().total / (1024**3)
    memoria_disponible = psutil.virtual_memory().available / (1024**3)
    print(f"💻 RAM total: {memoria_total:.1f} GB")
    print(f"💻 RAM disponible: {memoria_disponible:.1f} GB")

    # OPCIONES DE CHUNK SIZE (elige una):

    # Opción 1: Automático (recomendado para la mayoría de casos)
    # def procesar_archivo_por_chunks_seguro(archivo_sas: str, archivo_datos: str, archivo_salida: str, chunk_size: int = None):
    success = procesar_archivo_por_chunks_seguro(archivo_sas, archivo_datos, archivo_salida, chunk_size=400000)

    # Opción 2: Ultra conservador para sistemas con muy poca RAM
    # success = procesar_archivo_por_chunks_seguro(archivo_sas, archivo_datos, archivo_salida, chunk_size=5000)

    # Opción 3: Muy conservador
    # success = procesar_archivo_por_chunks_seguro(archivo_sas, archivo_datos, archivo_salida, chunk_size=10000)

    # Opción 4: Moderadamente conservador
    # success = procesar_archivo_por_chunks_seguro(archivo_sas, archivo_datos, archivo_salida, chunk_size=25000)

    if success:
        print("\n🎉 ¡Conversión completada exitosamente!")
    else:
        print("\n❌ Error en la conversión")

=== CONVERSOR SAS ULTRA-SEGURO ===
Diseñado específicamente para evitar errores de memoria
✓ Archivos encontrados
💻 RAM total: 62.4 GB
💻 RAM disponible: 48.6 GB
=== PROCESAMIENTO ULTRA-SEGURO POR CHUNKS ===
Archivo: ../data/2018/collection_sas/SAS_STU_FLT/FLT/CY07_MSU_FLT_TTM.SAS7BDAT

1. Extrayendo formatos y asignaciones...
   ✓ Formatos extraídos: 0
   ✓ Asignaciones extraídas: 744

Usando chunk size especificado: 400,000 filas

2. Iniciando procesamiento con chunks de 400,000 filas

--- Chunk 1 ---
Intentando leer desde fila 0, máximo 400,000 filas
Memoria disponible antes: 48.5 GB
✓ Chunk cargado: 107,174 filas × 751 columnas
✓ Memoria del chunk: 0.60 GB
Memoria disponible después: 47.6 GB
  Aplicando formatos al chunk...
  Formatos aplicados: 0
  Guardando chunk...
  ✓ Archivo CSV creado
✓ Total procesado hasta ahora: 107,174 filas
Memoria disponible después de limpiar: 48.5 GB
Último chunk procesado (menos filas de las solicitadas)

=== ✅ PROCESAMIENTO COMPLETADO ===
📁 Archivo f

In [7]:
import os

base = "../data/2018/collection_sas/SAS_STU_COG/COG/"

print("=== Archivos en la carpeta ===")
for f in sorted(os.listdir(base)):
    size_mb = os.path.getsize(os.path.join(base, f)) / (1024**2)
    print(f"  {f:50s} {size_mb:8.1f} MB")

=== Archivos en la carpeta ===
  CY07MSU_FMT_STU_COG.CPT                                 0.6 MB
  CY07MSU_FMT_STU_COG.SAS7BCAT                            0.6 MB
  CY07MSU_STU_COG.FORMAT.SAS                              0.1 MB
  Read Me.txt                                             0.0 MB
  cy07_msu_stu_cog.sas7bdat                           10837.0 MB


In [8]:
# Ver las primeras 60 líneas del FORMAT.SAS del 2018
path_format = "../data/2018/collection_sas/SAS_STU_COG/COG/CY07MSU_STU_COG.FORMAT.SAS"

with open(path_format, "r", encoding="utf-8", errors="replace") as f:
    for i, linea in enumerate(f):
        if i >= 60:
            break
        print(f"{i+1:3d}: {linea}", end="")

  1: %let DataDir =.;
  2: %let DataFile = CY07_MSU_STU_COG;
  3: libname DataLib "&DataDir.";
  4: OPTIONS FMTSEARCH=( DataLib.CY07MSU_Fmt_STU_COG );
  5: data &DataFile;
  6: set DataLib.&DataFile;
  7: Format ADMINMODE  ADMINMODENF.;
  8: Format BOOKID  BOOKIDNF.;
  9: Format CG121Q01A  MissF.;
 10: Format CG121Q01F  MissF.;
 11: Format CG121Q01S  Dichotomous2F.;
 12: Format CG121Q01SA  Dichotomous2F.;
 13: Format CG121Q01SB  Dichotomous2F.;
 14: Format CG121Q01SC  Dichotomous2F.;
 15: Format CG121Q01SD  Dichotomous2F.;
 16: Format CG121Q01SE  Dichotomous2F.;
 17: Format CG121Q01SF  Dichotomous2F.;
 18: Format CG121Q01SG  Dichotomous2F.;
 19: Format CG121Q01T  MissF.;
 20: Format CG121Q02A  MissF.;
 21: Format CG121Q02F  MissF.;
 22: Format CG121Q02S  Dichotomous2F.;
 23: Format CG121Q02SA  Dichotomous2F.;
 24: Format CG121Q02SB  Dichotomous2F.;
 25: Format CG121Q02SC  Dichotomous2F.;
 26: Format CG121Q02SD  Dichotomous2F.;
 27: Format CG121Q02T  MissF.;
 28: Format CG122Q01A  MissF

In [9]:
import pyreadstat

cat_path = "../data/2018/collection_sas/SAS_STU_COG/COG/CY07MSU_FMT_STU_COG.SAS7BCAT"

# Leer el catálogo de formatos
try:
    df_cat, meta_cat = pyreadstat.read_sas7bcat(cat_path)
    print("Columnas del catálogo:", df_cat.columns.tolist())
    print("Shape:", df_cat.shape)
    print(df_cat.head(20))
except Exception as e:
    print(f"Error: {e}")

Error: Unable to convert string to the requested encoding (invalid byte sequence)


In [10]:
import pyreadstat

cat_path = "../data/2018/collection_sas/SAS_STU_COG/COG/CY07MSU_FMT_STU_COG.SAS7BCAT"

# Probar diferentes encodings
for enc in ["latin1", "cp1252", "iso-8859-1", "utf-8"]:
    try:
        df_cat, meta_cat = pyreadstat.read_sas7bcat(cat_path, encoding=enc)
        print(f"✓ Encoding '{enc}' funcionó")
        print("Columnas:", df_cat.columns.tolist())
        print("Shape:", df_cat.shape)
        print(df_cat.head(10).to_string())
        break
    except Exception as e:
        print(f"✗ Encoding '{enc}': {e}")

✓ Encoding 'latin1' funcionó
Columnas: []
Shape: (0, 0)
Empty DataFrame
Columns: []
Index: []


In [11]:
# El .CPT podría tener los formatos en texto plano
cpt_path = "../data/2018/collection_sas/SAS_STU_COG/COG/CY07MSU_FMT_STU_COG.CPT"

for enc in ["utf-8", "latin1", "cp1252"]:
    try:
        with open(cpt_path, "r", encoding=enc) as f:
            contenido = f.read(3000)
        print(f"✓ Encoding '{enc}' funcionó para .CPT")
        print(contenido)
        break
    except Exception as e:
        print(f"✗ '{enc}': {e}")

✗ 'utf-8': 'utf-8' codec can't decode byte 0xbc in position 100: invalid start byte
✓ Encoding 'latin1' funcionó para .CPT
**COMPRESSED** **COMPRESSED** **COMPRESSED** **COMPRESSED** **COMPRESSED********LIB CONTROL W32_10PR¼ SAS9.4¼I¼¥C701MY32¼CY07MSU_FMT_STU_COG¼OC0  74¼MAX LRECL =147¼
OBJECT CONTROL ADMINMODENF¼FORMAT¼10¼Ì10¼4096¼O0010001000080000¼05NOV19:10:38:32  05NOV19:10:38:32SPAN CONTROL 3¼113¼2¼33¼L¼113¼ADMINMODHpH¨&è   (¼¼¼ ¼P7-ê ¼
  ¼  ADMINMODENF¼A  ¼­SPAN CONTROL 4¼24¼
1¼24¼L¼24¼ ¼A¼A¼¼¶SPAN CONTROL 4¼24¼
1¼24¼L¼24¼ ¼A ¼A ¼¼¶SPAN CONTROL 5¼9¼1¼9¼L¼9¼  Paper¼ÅSPAN CONTROL 5¼12¼
1¼12¼L¼12¼ Computer¼ÂEnd of Object¼ÁOBJECT CONTROL BOOKIDNF¼FORMAT¼10¼Ì10¼4096¼O0010001000070000¼05NOV19:10:38:32  05NOV19:10:38:32SPAN CONTROL 3¼113¼2¼33¼L¼113¼BOOKIDNFHpH¨   (  ¼U¼U  P  
H7-ê ¼
   ¼A ADMINMODENF¼A  ¼­SPAN CONTROL 4¼24¼
1¼24¼L¼24¼ ¼A¼A¼¼¶SPAN CONTROL 4¼24¼
